In [1]:
import json
import os

# Load Kaggle API credentials from kaggle_key.json
with open('kaggle_key.json', 'r') as f:
    kaggle_credentials = json.load(f)

# Set Kaggle API credentials as environment variables
os.environ['KAGGLE_USERNAME'] = kaggle_credentials['username']
os.environ['KAGGLE_KEY'] = kaggle_credentials['key']

print("Kaggle API credentials loaded successfully!")
print(f"Username: {kaggle_credentials['username']}")
print("Key: ****" + kaggle_credentials['key'][-4:])  # Show only last 4 characters for security

Kaggle API credentials loaded successfully!
Username: cubbic
Key: ****f9db


In [2]:

os.environ["KERAS_BACKEND"] = "jax"
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"]="1.00"

import keras_hub
# Load the base model
gemma_lm = keras_hub.models.Gemma3CausalLM.from_preset("gemma3_instruct_4b_text")

# Enable LoRA with the same rank as during training
gemma_lm.backbone.enable_lora(rank=4) # type: ignore

# Load the trained LoRA weights
gemma_lm.backbone.load_lora_weights("trained_gemma_4b_disaster_lora.lora.h5") # type: ignore

# Compile with the same sampler for inference
sampler = keras_hub.samplers.GreedySampler()
gemma_lm.compile(sampler=sampler)




2025-08-06 22:37:14.469697: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1754512634.485588    7017 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1754512634.490803    7017 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1754512634.503883    7017 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1754512634.503901    7017 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1754512634.503904    7017 computation_placer.cc:177] computation placer alr

In [3]:
import pandas as pd

test = pd.read_csv("data/test.csv")

In [4]:
template = "Tweet: {tweet}\nIs this about a real disaster? Answer Yes or No\nAnswer:"
print(len(test))

submission = pd.DataFrame(columns=['id', 'target'])
submission['id'] = test['id']

predictions = []
for i, row in test.iterrows():
    prompt = template.format(tweet=row['text'])
    response = gemma_lm.generate(prompt)
    last_3_chars = response[-3:]
    answer = -1
    if("answer:yes" in response.lower()):
        answer = 1
    elif("answer:no" in response.lower()):
        answer = 0
    predictions.append(answer)

    if(answer == -1):
        print(f"Row {i}/{len(test)}: No answer found in response: {response}")
    
    print(f"Row {i}/{len(test)}: {answer}")

submission['target'] = predictions

submission.to_csv('data/submission_gemma_4b.csv', index=False)

3263
Row 0/3263: 0
Row 1/3263: 1
Row 2/3263: 1
Row 3/3263: 1
Row 4/3263: 1
Row 5/3263: 1
Row 6/3263: 0
Row 7/3263: 0
Row 8/3263: 0
Row 9/3263: 0
Row 10/3263: 0
Row 11/3263: 0
Row 12/3263: 0
Row 13/3263: 0
Row 14/3263: 0
Row 15/3263: 1
Row 16/3263: 0
Row 17/3263: 0
Row 18/3263: 0
Row 19/3263: 0
Row 20/3263: 0
Row 21/3263: 0
Row 22/3263: 0
Row 23/3263: 1
Row 24/3263: 0
Row 25/3263: 0
Row 26/3263: 0
Row 27/3263: 0
Row 28/3263: 0
Row 29/3263: 1
Row 30/3263: 0
Row 31/3263: 0
Row 32/3263: 0
Row 33/3263: 0
Row 34/3263: 1
Row 35/3263: 0
Row 36/3263: 0
Row 37/3263: 0
Row 38/3263: 0
Row 39/3263: 1
Row 40/3263: 0
Row 41/3263: 1
Row 42/3263: 0
Row 43/3263: 1
Row 44/3263: 0
Row 45/3263: 0
Row 46/3263: 0
Row 47/3263: 0
Row 48/3263: 1
Row 49/3263: 0
Row 50/3263: 0
Row 51/3263: 0
Row 52/3263: 1
Row 53/3263: 0
Row 54/3263: 0
Row 55/3263: 0
Row 56/3263: 0
Row 57/3263: 0
Row 58/3263: 0
Row 59/3263: 0
Row 60/3263: 1
Row 61/3263: 0
Row 62/3263: 1
Row 63/3263: 0
Row 64/3263: 1
Row 65/3263: 1
Row 66/3263: 0
